In [2]:
import sys, json
sys.path.append("..")
%load_ext autoreload
%autoreload 2

import torch
from torch.utils.data import DataLoader
from src.tokenizer import CharTokenizer
from src.dataset import SQLDataset, collate
from src.model import GPTConfig, TinyGPT

def load(name):
    return [json.loads(line) for line in open(f"../data/raw/{name}.jsonl", encoding="utf-8")]

train = load("train")
tok = CharTokenizer.build([r["question"] for r in train] + [r["sql"] for r in train])
print("vocab size:", tok.vocab_size)

vocab size: 73


In [3]:
config = GPTConfig(vocab_size=tok.vocab_size)
model = TinyGPT(config)

loader = DataLoader(SQLDataset(train, tok), batch_size=4, shuffle=True,
                    collate_fn=lambda b: collate(b, tok.pad_id))
xb, yb = next(iter(loader))

out = model(xb)
print("input :", xb.shape)
print("output:", out.shape)

n_params = sum(p.numel() for p in model.parameters())
print("parameters:", n_params)

input : torch.Size([4, 253])
output: torch.Size([4, 253, 128])
parameters: 44160


In [4]:
from src.model import Head

config = GPTConfig(vocab_size=73)
head = Head(config, head_size=32)

# 1. Shapes
x = torch.randn(2, 10, 128)
print("output:", head(x).shape)                 # expect [2, 10, 32]

# 2. The causality test: change ONLY the last token, check who notices
head.eval()                                     # dropout off so results are repeatable
x = torch.randn(1, 8, 128)
out1 = head(x)
x2 = x.clone()
x2[:, -1, :] = torch.randn(128)                 # replace the last token with something else
out2 = head(x2)

print("earlier tokens unchanged:", torch.allclose(out1[:, :-1], out2[:, :-1]))   # expect True
print("last token changed:      ", not torch.allclose(out1[:, -1], out2[:, -1]))  # expect True

output: torch.Size([2, 10, 32])
earlier tokens unchanged: True
last token changed:       True


In [5]:
import sys, json, math
sys.path.append("..")
%load_ext autoreload
%autoreload 2

import torch
from src.tokenizer import CharTokenizer
from src.dataset import SQLDataset, collate
from src.model import GPTConfig, TinyGPT

def load(name):
    return [json.loads(line) for line in open(f"../data/raw/{name}.jsonl", encoding="utf-8")]

train = load("train")
tok = CharTokenizer.build([r["question"] for r in train] + [r["sql"] for r in train])

ds = SQLDataset(train, tok)
xb, yb = collate([ds[i] for i in range(4)], tok.pad_id)   # the same 4 examples every time
print("vocab size:", tok.vocab_size, "| batch:", xb.shape)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
vocab size: 73 | batch: torch.Size([4, 253])


In [6]:
torch.manual_seed(0)
model = TinyGPT(GPTConfig(vocab_size=tok.vocab_size))

print("parameters:", sum(p.numel() for p in model.parameters()))   # expect 845312

logits, loss = model(xb, yb)
print("logits:", logits.shape)
print("loss:", round(loss.item(), 4), "| random-guess level:", round(math.log(tok.vocab_size), 4))

parameters: 845312
logits: torch.Size([4, 253, 73])
loss: 4.3248 | random-guess level: 4.2905


In [7]:
model.eval()
x = xb[:1, :50].clone()
l1, _ = model(x)

x2 = x.clone()
x2[0, -1] = (x2[0, -1] + 1) % tok.vocab_size      # change only the last token
l2, _ = model(x2)

print("earlier positions unchanged:", torch.allclose(l1[:, :-1], l2[:, :-1], atol=1e-5))
print("last position changed:      ", not torch.allclose(l1[:, -1], l2[:, -1]))

earlier positions unchanged: True
last position changed:       True


In [8]:
torch.manual_seed(0)
model = TinyGPT(GPTConfig(vocab_size=tok.vocab_size, dropout=0.0))
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)

model.train()
for step in range(201):
    logits, loss = model(xb, yb)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 40 == 0:
        print(step, round(loss.item(), 4))

0 4.3344
40 0.3045
80 0.0407
120 0.0203
160 0.013
200 0.0092


In [9]:
model.eval()
for i in range(4):
    row = train[i]
    prompt = torch.tensor([tok.encode(row["question"]) + [tok.sep_id]])
    out = model.generate(prompt, max_new_tokens=300, eos_id=tok.eos_id)
    predicted = tok.decode(out[0, prompt.shape[1]:].tolist())
    print(predicted == row["sql"], "|", predicted[:70])

True | SELECT AVG(price) FROM products WHERE category = 'electronics'
True | SELECT AVG(price) FROM products WHERE category = 'stationery'
True | SELECT AVG(price) FROM products WHERE category = 'toys'
True | SELECT COUNT(*) FROM orders JOIN customers ON orders.customer_id = cus


In [10]:
import json
import matplotlib.pyplot as plt

run = "smoke"
history = json.load(open(f"../results/{run}_history.json"))
steps = [h["step"] for h in history]

plt.plot(steps, [h["train_loss"] for h in history], marker="o", label="train")
plt.plot(steps, [h["val_loss"] for h in history], marker="o", label="validation")
plt.xlabel("step")
plt.ylabel("loss")
plt.title(f"Loss curves ({run})")
plt.legend()
plt.savefig(f"../results/loss_curve_{run}.png", dpi=150, bbox_inches="tight")
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: '../results/smoke_history.json'